In [8]:


import os, json, warnings, importlib.util, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import joblib

warnings.filterwarnings("ignore")

REPO = Path.cwd()
for _ in range(8):
    if (REPO / "homework").exists() or (REPO / ".git").exists():
        break
    REPO = REPO.parent

H13  = REPO / "homework" / "homework13"
H11  = REPO / "homework" / "homework11"
H12  = REPO / "homework" / "homework12"
DATA = H13 / "data"
SRC  = H13 / "src"
NB   = H13 / "notebooks"
ART  = H13 / "artifacts"
IMG  = ART / "images"
REPS = H13 / "reports"
MODEL_DIR = H13 / "model"

for d in [DATA/"raw", DATA/"processed", SRC, NB, ART, IMG, REPS, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)


def discover_csv():
    cands = []
    for base in [H11, H12, REPO]:
        if base.exists():
            cands += list(base.rglob("*.csv"))
    for p in cands:
        try:
            df = pd.read_csv(p)
            if df.select_dtypes(include=[np.number]).shape[1] >= 2 and df.shape[0] >= 120:
                return p, df
        except Exception:
            pass
    return None, None

path, df = discover_csv()
if path is None:
    rng = np.random.default_rng(42)
    n = 400
    risk = rng.normal(50, 12, n)
    mkt  = rng.normal(100, 40, n)
    rev  = 20000 + 110*risk + 8*mkt + rng.normal(0, 2500, n)
    df = pd.DataFrame({"risk_index_var95": risk, "marketing_spend": mkt, "revenue": rev})
    path = DATA / "processed" / "synthetic_stage13.csv"
    df.to_csv(path, index=False)
else:
    # ensure at least 2 features after dropping target
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    tgt = "revenue" if "revenue" in df.columns else (num_cols[-1] if num_cols else df.columns[-1])
    feats = [c for c in num_cols if c != tgt]
    if len(feats) < 2:
        df["synthetic_aux"] = np.random.default_rng(0).normal(0,1,len(df))
    path = DATA / "processed" / "discovered_for_stage13.csv"
    df.to_csv(path, index=False)

TARGET = "revenue" if "revenue" in df.columns else df.select_dtypes(include=[np.number]).columns[-1]
X = df.select_dtypes(include=[np.number]).drop(columns=[TARGET], errors="ignore").copy()
y = df[TARGET].copy()

# Guarantee at least 2 feature columns for path-based predict
if X.shape[1] < 2:
    X["synthetic_aux"] = np.random.default_rng(0).normal(0,1,len(X))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=13)

pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("lr", LinearRegression())
]).fit(X_train, y_train)

yhat = pipe.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, yhat)))

MODEL_PATH = MODEL_DIR / "model.pkl"
joblib.dump(pipe, MODEL_PATH)

metadata = {
    "target": TARGET,
    "feature_names": X.columns.tolist(),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "rmse_test": rmse
}
(MODEL_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

# Plot for /plot
plt.figure()
mn = float(min(y_test.min(), yhat.min()))
mx = float(max(y_test.max(), yhat.max()))
plt.scatter(y_test, yhat, s=18, alpha=0.7)
plt.plot([mn, mx], [mn, mx], ls="--")
plt.xlabel("Actual"); plt.ylabel("Predicted"); plt.title("Predicted vs Actual")
plt.tight_layout(); plt.savefig(IMG / "pred_vs_actual.png", dpi=160); plt.close()

print(f"[model] Saved: {MODEL_PATH} | rmse={rmse:.1f}")
print("[assets] pred_vs_actual ->", IMG / "pred_vs_actual.png")


app_py = '''import io, json
from pathlib import Path

import numpy as np
import pandas as pd
from flask import Flask, request, jsonify, send_file
import joblib
import matplotlib.pyplot as plt

HERE = Path(__file__).resolve().parents[1]
MODEL_PATH = HERE / "model" / "model.pkl"
META_PATH  = HERE / "model" / "metadata.json"
ART_IMG    = HERE / "artifacts" / "images" / "pred_vs_actual.png"

app = Flask(__name__)

# Load model & metadata once
model = joblib.load(MODEL_PATH)
meta  = json.loads(META_PATH.read_text(encoding="utf-8"))
FEATURES = meta.get("feature_names", [])

def _coerce_df(payload):
    """Accept dict or list-of-dicts; coerce to DataFrame with model feature columns."""
    if isinstance(payload, dict):
        df = pd.DataFrame([payload])
    elif isinstance(payload, list):
        df = pd.DataFrame(payload)
    else:
        raise ValueError("Payload must be an object or list of objects")
    # ensure expected columns exist and are ordered; imputer will handle NaN
    for c in FEATURES:
        if c not in df.columns:
            df[c] = np.nan
    df = df[FEATURES]
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

@app.get("/health")
def health():
    return jsonify(status="ok", model=str(MODEL_PATH.name), features=FEATURES, target=meta.get("target"))

@app.post("/predict")
def predict_post():
    try:
        payload = request.get_json(force=True, silent=False)
        if isinstance(payload, dict) and "records" in payload:
            df = _coerce_df(payload["records"])
        else:
            df = _coerce_df(payload)  # support single record or list
        preds = model.predict(df).tolist()
        return jsonify(predictions=preds, n=len(preds))
    except Exception as e:
        return jsonify(error=str(e)), 400

@app.get("/predict/<x1>")
@app.get("/predict/<x1>/<x2>")
def predict_path(x1, x2=None):
    try:
        if not FEATURES:
            return jsonify(error="Model feature list is empty"), 400
        row = {FEATURES[0]: float(x1)}
        if x2 is not None and len(FEATURES) > 1:
            row[FEATURES[1]] = float(x2)
        df = _coerce_df(row)
        pred = float(model.predict(df)[0])
        return jsonify(prediction=pred)
    except Exception as e:
        return jsonify(error=str(e)), 400

@app.get("/plot")
def plot_png():
    if ART_IMG.exists():
        return send_file(ART_IMG, mimetype="image/png")
    buf = io.BytesIO()
    xs = np.linspace(0, 10, 100); ys = xs + np.random.normal(0, 1, 100)
    plt.figure(); plt.scatter(xs, ys, s=16, alpha=0.7)
    plt.plot([0,10],[0,10], "--"); plt.tight_layout()
    plt.savefig(buf, format="png", dpi=144); plt.close(); buf.seek(0)
    return send_file(buf, mimetype="image/png")

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=8000, debug=False)
'''
(SRC / "app.py").write_text(app_py, encoding="utf-8")
print("[api] Wrote:", SRC / "app.py")


(H13 / "requirements.txt").write_text(
    "flask\npandas\nnumpy\nscikit-learn\njoblib\nmatplotlib\nrequests\n",
    encoding="utf-8"
)
print("[deps] Wrote:", H13 / "requirements.txt")


report_text = """# Stage 13 — Productization

**Included**
- Pickled model: `model/model.pkl` (+ `model/metadata.json`)
- Flask API: `src/app.py` with `/health`, `POST /predict`, `GET /predict/<x1>[/<x2>]`, `/plot`
- `requirements.txt`
- Pred vs Actual image: `artifacts/images/pred_vs_actual.png`

**Run**
1. `pip install -r requirements.txt`
2. `python src/app.py` → http://127.0.0.1:8000
"""
(REPS / "stage13_productization.md").write_text(report_text, encoding="utf-8")

readme_text = """# Homework 13 — Productization

## Layout
- data/raw, data/processed
- src/app.py (Flask API)
- notebooks/stage13_productization.ipynb
- artifacts/images/pred_vs_actual.png
- reports/stage13_productization.md
- model/model.pkl, model/metadata.json
- requirements.txt

## Run the API
pip install -r requirements.txt
python src/app.py   # http://127.0.0.1:8000
"""
(H13 / "README.md").write_text(readme_text, encoding="utf-8")
print("[docs] Wrote README and report.")

try:
    spec = importlib.util.spec_from_file_location("api_app", SRC / "app.py")
    api_app = importlib.util.module_from_spec(spec)
    sys.modules["api_app"] = api_app
    spec.loader.exec_module(api_app)
    client = api_app.app.test_client()
    r_health = client.get("/health")
    r_plot   = client.get("/plot")
    print("[smoke] /health:", r_health.status_code, "| /plot:", r_plot.status_code, "bytes:", len(r_plot.data))
except Exception as e:
    print("[smoke] skipped:", e)

print("To run the API:\n  pip install -r requirements.txt\n  python src/app.py  # http://127.0.0.1:8000")


[model] Saved: C:\Users\User\bootcamp_Khushi_Khanna\homework\homework13\model\model.pkl | rmse=2766.2
[assets] pred_vs_actual -> C:\Users\User\bootcamp_Khushi_Khanna\homework\homework13\artifacts\images\pred_vs_actual.png
[api] Wrote: C:\Users\User\bootcamp_Khushi_Khanna\homework\homework13\src\app.py
[deps] Wrote: C:\Users\User\bootcamp_Khushi_Khanna\homework\homework13\requirements.txt
[docs] Wrote README and report.
[smoke] /health: 200 | /plot: 200 bytes: 54086
To run the API:
  pip install -r requirements.txt
  python src/app.py  # http://127.0.0.1:8000
